In [1]:
# Imports nécessaires
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Charger la dataframe finale
df = pd.read_csv(r"C:\\Users\\zizou\\OneDrive\\Desktop\\stage 3ème\\day 2\\csvfiles\\dataframefinale.csv", sep=';', encoding='utf-8-sig')

# Vérification des types de colonnes
print("Types de colonnes avant prétraitement :\n", df.dtypes)

# Prétraitement de dataloadingdate
if 'dataloadingdate' in df.columns:
    df['dataloadingdate'] = pd.to_datetime(df['dataloadingdate'], errors='coerce')  # Convertir en datetime
    df['year'] = df['dataloadingdate'].dt.year  # Extraire uniquement l'année
    df = df.drop(columns=['dataloadingdate'])  # Supprimer la colonne originale

# Éliminer les features catégorielles spécifiées
columns_to_drop = ['joursemaine', 'ISIN', 'Libellé', 'JourSemaine']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# Vérification et suppression des NaN
print("Nombre de NaN par colonne avant suppression :\n", df.isna().sum())
df = df.dropna()  # Supprimer les lignes avec NaN
print("Nombre de NaN par colonne après suppression :\n", df.isna().sum())

# Préparation : Séparer features et target
X = df.drop(columns=['Montant'])  # Features
y = df['Montant']  # Target

# Identifier colonnes catégorielles et numériques (categorical_cols devrait être vide maintenant)
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()  # Colonnes de type string
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()  # Toutes les colonnes restantes

# Vérification des colonnes
print("Colonnes catégorielles :", categorical_cols)  # Devrait être vide
print("Colonnes numériques :", numerical_cols)

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Données prêtes : Train shape", X_train.shape, "Test shape", X_test.shape)

Types de colonnes avant prétraitement :
 dataloadingdate      object
Jour                  int64
Mois                  int64
NumeroSemaine         int64
Trimestre             int64
JourSemaineNum        int64
JourSemaine          object
ISIN                 object
Libellé              object
Nombre de Titres    float64
Montant             float64
Echéance            float64
Taux                float64
dtype: object
Nombre de NaN par colonne avant suppression :
 Jour                    0
Mois                    0
NumeroSemaine           0
Trimestre               0
JourSemaineNum          0
Nombre de Titres        0
Montant                 0
Echéance                0
Taux                    0
year                11218
dtype: int64
Nombre de NaN par colonne après suppression :
 Jour                0
Mois                0
NumeroSemaine       0
Trimestre           0
JourSemaineNum      0
Nombre de Titres    0
Montant             0
Echéance            0
Taux                0
year            

In [2]:
# Fonction pour calculer RMSE
def calculate_rmse(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return rmse

# Préprocesseur de base (seulement pour colonnes numériques, pas de catégorielles)
def get_preprocessor(with_scaling=False):
    transformers = [
        ('num', 'passthrough' if not with_scaling else StandardScaler(), numerical_cols)
    ]
    return ColumnTransformer(transformers=transformers)

# Pour feature selection
def get_pipeline_with_fs(model, k=10):
    preprocessor = get_preprocessor(with_scaling=True)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', SelectKBest(f_regression, k=min(k, len(numerical_cols)))),
        ('model', model)
    ])

In [3]:
print("=== Modèle 1: Régression Linéaire ===")

# Étape 1: Modèle de base
lr_base = Pipeline([('preprocessor', get_preprocessor()), ('model', LinearRegression())])
rmse_base = calculate_rmse(lr_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
lr_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', LinearRegression())])
rmse_scaled = calculate_rmse(lr_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
lr_fs = get_pipeline_with_fs(LinearRegression(), k=10)
rmse_fs = calculate_rmse(lr_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning (utilisation de Ridge pour régularisation)
param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
lr_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', Ridge())])
grid_search_lr = GridSearchCV(lr_tuned, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search_lr.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_lr.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning (Ridge):", rmse_tuned, "Meilleurs params:", grid_search_lr.best_params_)

=== Modèle 1: Régression Linéaire ===
Étape 1 - RMSE base: 6.636464313540332
Étape 2 - RMSE avec normalisation: 6.636464313540366
Étape 3 - RMSE avec feature selection: 6.636464313540366
Étape 4 - RMSE après fine-tuning (Ridge): 6.635261888794809 Meilleurs params: {'model__alpha': 10.0}


In [4]:
print("=== Modèle 2: Random Forest Regressor ===")

# Étape 1: Modèle de base
rf_base = Pipeline([('preprocessor', get_preprocessor()), ('model', RandomForestRegressor(random_state=42))])
rmse_base = calculate_rmse(rf_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
rf_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', RandomForestRegressor(random_state=42))])
rmse_scaled = calculate_rmse(rf_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
rf_fs = get_pipeline_with_fs(RandomForestRegressor(random_state=42), k=10)
rmse_fs = calculate_rmse(rf_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__n_estimators': [100, 200, 300], 'model__max_depth': [10, 20, None], 'model__min_samples_split': [2, 5, 10]}
rf_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', RandomForestRegressor(random_state=42))])
grid_search_rf = GridSearchCV(rf_tuned, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search_rf.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_rf.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_rf.best_params_)

=== Modèle 2: Random Forest Regressor ===
Étape 1 - RMSE base: 3.957690685042229
Étape 2 - RMSE avec normalisation: 3.9753118122876017
Étape 3 - RMSE avec feature selection: 3.9753118122876017
Étape 4 - RMSE après fine-tuning: 3.907189312947126 Meilleurs params: {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 300}


In [5]:
print("=== Modèle 3: XGBoost Regressor ===")

# Étape 1: Modèle de base
xgb_base = Pipeline([('preprocessor', get_preprocessor()), ('model', XGBRegressor(random_state=42))])
rmse_base = calculate_rmse(xgb_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
xgb_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', XGBRegressor(random_state=42))])
rmse_scaled = calculate_rmse(xgb_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
xgb_fs = get_pipeline_with_fs(XGBRegressor(random_state=42), k=10)
rmse_fs = calculate_rmse(xgb_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__n_estimators': [100, 200, 300], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__max_depth': [3, 5, 7]}
xgb_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', XGBRegressor(random_state=42))])
grid_search_xgb = GridSearchCV(xgb_tuned, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search_xgb.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_xgb.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_xgb.best_params_)

=== Modèle 3: XGBoost Regressor ===
Étape 1 - RMSE base: 4.1239511657396495
Étape 2 - RMSE avec normalisation: 4.1239511657396495
Étape 3 - RMSE avec feature selection: 4.1239511657396495
Étape 4 - RMSE après fine-tuning: 4.266348639309649 Meilleurs params: {'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 300}


In [6]:
print("=== Modèle 4: LightGBM Regressor ===")

# Étape 1: Modèle de base
lgbm_base = Pipeline([('preprocessor', get_preprocessor()), ('model', LGBMRegressor(random_state=42, verbose=-1))])
rmse_base = calculate_rmse(lgbm_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
lgbm_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', LGBMRegressor(random_state=42, verbose=-1))])
rmse_scaled = calculate_rmse(lgbm_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
lgbm_fs = get_pipeline_with_fs(LGBMRegressor(random_state=42, verbose=-1), k=10)
rmse_fs = calculate_rmse(lgbm_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__n_estimators': [100, 200, 300], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__max_depth': [3, 5, 7], 'model__num_leaves': [15, 31, 63]}
lgbm_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', LGBMRegressor(random_state=42, verbose=-1))])
grid_search_lgbm = GridSearchCV(lgbm_tuned, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search_lgbm.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_lgbm.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_lgbm.best_params_)

=== Modèle 4: LightGBM Regressor ===
Étape 1 - RMSE base: 3.768532060187388
Étape 2 - RMSE avec normalisation: 3.744220073271282
Étape 3 - RMSE avec feature selection: 3.744220073271282
Étape 4 - RMSE après fine-tuning: 3.863512962102838 Meilleurs params: {'model__learning_rate': 0.1, 'model__max_depth': 7, 'model__n_estimators': 300, 'model__num_leaves': 63}


In [7]:
print("=== Modèle 5: CatBoost Regressor ===")

# Étape 1: Modèle de base
preprocessor_cat = ColumnTransformer([
    ('num', 'passthrough', numerical_cols)
])
cat_base = Pipeline([
    ('preprocessor', preprocessor_cat),
    ('model', CatBoostRegressor(random_state=42, verbose=0))
])
rmse_base = calculate_rmse(cat_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
preprocessor_cat_scaled = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols)
])
cat_scaled = Pipeline([
    ('preprocessor', preprocessor_cat_scaled),
    ('model', CatBoostRegressor(random_state=42, verbose=0))
])
rmse_scaled = calculate_rmse(cat_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
cat_fs = Pipeline([
    ('preprocessor', get_preprocessor(with_scaling=True)),
    ('feature_selection', SelectKBest(f_regression, k=10)),
    ('model', CatBoostRegressor(random_state=42, verbose=0))
])
rmse_fs = calculate_rmse(cat_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__iterations': [100, 200, 300], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__depth': [3, 5, 7]}
cat_tuned = Pipeline([
    ('preprocessor', preprocessor_cat_scaled),
    ('model', CatBoostRegressor(random_state=42, verbose=0))
])
grid_search_cat = GridSearchCV(cat_tuned, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search_cat.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_cat.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_cat.best_params_)

=== Modèle 5: CatBoost Regressor ===
Étape 1 - RMSE base: 3.7303187530902338
Étape 2 - RMSE avec normalisation: 3.7303199972146976


c:\Users\zizou\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:776: UserWarning: k=10 is greater than n_features=9. All the features will be returned.
  warnings.warn(


Étape 3 - RMSE avec feature selection: 3.7303199972146976
Étape 4 - RMSE après fine-tuning: 3.73844987249109 Meilleurs params: {'model__depth': 7, 'model__iterations': 300, 'model__learning_rate': 0.1}


In [8]:
print("=== Modèle 6: Gradient Boosting Regressor ===")

# Étape 1: Modèle de base
gb_base = Pipeline([('preprocessor', get_preprocessor()), ('model', GradientBoostingRegressor(random_state=42))])
rmse_base = calculate_rmse(gb_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
gb_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', GradientBoostingRegressor(random_state=42))])
rmse_scaled = calculate_rmse(gb_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
gb_fs = get_pipeline_with_fs(GradientBoostingRegressor(random_state=42), k=10)
rmse_fs = calculate_rmse(gb_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__n_estimators': [100, 200, 300], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__max_depth': [3, 5, 7], 'model__min_samples_split': [2, 5, 10]}
gb_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', GradientBoostingRegressor(random_state=42))])
grid_search_gb = GridSearchCV(gb_tuned, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search_gb.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_gb.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_gb.best_params_)

=== Modèle 6: Gradient Boosting Regressor ===
Étape 1 - RMSE base: 4.2176253035452
Étape 2 - RMSE avec normalisation: 4.221534899280053
Étape 3 - RMSE avec feature selection: 4.221534899280053
Étape 4 - RMSE après fine-tuning: 4.159599568514592 Meilleurs params: {'model__learning_rate': 0.05, 'model__max_depth': 7, 'model__min_samples_split': 10, 'model__n_estimators': 300}


In [9]:
# Comparaison : Collecte tous les RMSE finaux
rmse_results = {
    'Linear Regression': np.sqrt(mean_squared_error(y_test, grid_search_lr.predict(X_test))),
    'Random Forest': np.sqrt(mean_squared_error(y_test, grid_search_rf.predict(X_test))),
    'XGBoost': np.sqrt(mean_squared_error(y_test, grid_search_xgb.predict(X_test))),
    'LightGBM': np.sqrt(mean_squared_error(y_test, grid_search_lgbm.predict(X_test))),
    'CatBoost': np.sqrt(mean_squared_error(y_test, grid_search_cat.predict(X_test))),
    'Gradient Boosting': np.sqrt(mean_squared_error(y_test, grid_search_gb.predict(X_test)))
}
best_model = min(rmse_results, key=rmse_results.get)
print("Meilleur modèle:", best_model, "avec RMSE:", rmse_results[best_model])


Meilleur modèle: CatBoost avec RMSE: 3.73844987249109
